# 01 — Buffering Points

The degree-offset circle from notebook 00 is a useful sketch — but it is geometrically wrong. At latitude 35°N, one degree of longitude is only ~91 km, not 111 km. The result is an oval, not a circle.

This notebook builds an accurate point buffer using the **destination point formula** from module 06: given a center, a bearing, and a distance in kilometers, compute the exact location. Sample that at every bearing from 0° to 360° and you have a true circular buffer.

## Setup

In [1]:
import json
import math
from pathlib import Path
from ipyleaflet import Map, GeoJSON, basemaps

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

## The Destination Point Formula

Given a start point `(lon, lat)`, a bearing in degrees (0° = North, clockwise), and a distance in kilometers, the destination point formula returns the `[lon, lat]` of the point that lies exactly that far away in that direction.

This is the inverse of haversine: haversine measures *how far* two points are; destination point finds *where* a point is.

```
center + bearing + distance  →  point on the circle
```

Repeat for every bearing from 0° to 360° and you trace the full circumference.

In [2]:
def destination_point(lon, lat, bearing_deg, distance_km):
    """
    Given a start point and a bearing + distance, return the destination [lon, lat].
    Uses the spherical Earth model (R = 6371 km).
    """
    R     = 6371.0
    d     = distance_km / R
    phi1  = math.radians(lat)
    lam1  = math.radians(lon)
    theta = math.radians(bearing_deg)

    phi2 = math.asin(
        math.sin(phi1) * math.cos(d) +
        math.cos(phi1) * math.sin(d) * math.cos(theta)
    )
    lam2 = lam1 + math.atan2(
        math.sin(theta) * math.sin(d) * math.cos(phi1),
        math.cos(d) - math.sin(phi1) * math.sin(phi2)
    )
    return [math.degrees(lam2), math.degrees(phi2)]


# Quick check: 111 km due north of the equator should reach ~1° of latitude
result = destination_point(0, 0, 0, 111)
print(f"111 km north of (0, 0):  lon={result[0]:.4f}, lat={result[1]:.4f}")
print(f"Expected: lon≈0, lat≈1")

111 km north of (0, 0):  lon=0.0000, lat=0.9982
Expected: lon≈0, lat≈1


## Building a Point Buffer

Sample `n_points` evenly around the full 360° compass, computing the destination at `radius_km` in each direction. Connect those points into a closed polygon ring.

In [3]:
def point_buffer(lon, lat, radius_km, n_points=64):
    """
    Return a GeoJSON Polygon geometry approximating a circle of
    radius_km around (lon, lat). Uses n_points vertices.
    """
    ring = []
    for i in range(n_points):
        bearing = 360.0 * i / n_points
        ring.append(destination_point(lon, lat, bearing, radius_km))
    ring.append(ring[0])   # close the ring
    return {"type": "Polygon", "coordinates": [ring]}


def buffer_feature(lon, lat, radius_km, name="", n_points=64):
    """Wrap point_buffer in a GeoJSON Feature with properties."""
    return {
        "type": "Feature",
        "properties": {"name": name, "radius_km": radius_km},
        "geometry": point_buffer(lon, lat, radius_km, n_points)
    }


# Test: 100 km buffer around Tehran
tehran = [51.388, 35.695]
buf = point_buffer(*tehran, radius_km=100)
print("Geometry type: ", buf["type"])
print("Ring points:   ", len(buf["coordinates"][0]))
print("First point:   ", buf["coordinates"][0][0])
print("Last point:    ", buf["coordinates"][0][-1], "  (should match first)")

Geometry type:  Polygon
Ring points:    65
First point:    [51.388000000000005, 36.59432160591873]
Last point:     [51.388000000000005, 36.59432160591873]   (should match first)


## Visualizing a Single Buffer

In [4]:
buf_fc = {
    "type": "FeatureCollection",
    "features": [
        buffer_feature(*tehran, radius_km=150, name="150 km zone"),
        {
            "type": "Feature",
            "properties": {"name": "Tehran"},
            "geometry": {"type": "Point", "coordinates": tehran}
        }
    ]
}

m = Map(center=(tehran[1], tehran[0]), zoom=5, basemap=basemaps.CartoDB.Positron)
m.add(GeoJSON(
    data=buf_fc,
    style={"color": "#e74c3c", "fillColor": "#e74c3c", "fillOpacity": 0.25, "weight": 2}
))
m

Map(center=[35.695, 51.388], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom…

## Multiple Buffers from One Point

Stack several buffers at different radii to create **concentric zones**. Each ring represents a different level of proximity — the spatial equivalent of nested threshold values.

In [5]:
radii = [50, 150, 300]   # km
colors = ["#e74c3c", "#e67e22", "#f1c40f"]

# Build outermost → innermost so smaller rings render on top
ring_features = []
for radius, color in reversed(list(zip(radii, colors))):
    ring_features.append(buffer_feature(*tehran, radius_km=radius, name=f"{radius} km"))

m2 = Map(center=(tehran[1], tehran[0]), zoom=4, basemap=basemaps.CartoDB.Positron)

for feat, color in zip(reversed(ring_features), colors):
    m2.add(GeoJSON(
        data={"type": "FeatureCollection", "features": [feat]},
        style={"color": color, "fillColor": color, "fillOpacity": 0.3, "weight": 1.5}
    ))
m2

Map(center=[35.695, 51.388], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom…

## Saving the Module's Target Dataset

The target points used throughout this module are the destination cities from the module 09 paths. We save them here for consistent reuse across notebooks 02–06.

In [6]:
targets = [
    {"name": "Tehran",   "lon":  51.388, "lat":  35.695, "path": "Alpha"},
    {"name": "Honolulu", "lon": -157.855, "lat":  21.305, "path": "Bravo"},
    {"name": "Madrid",   "lon":  -3.703, "lat":  40.417, "path": "Charlie"},
    {"name": "Riyadh",   "lon":  46.675, "lat":  24.688, "path": "Delta"},
]

targets_fc = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {"name": t["name"], "path": t["path"]},
            "geometry": {"type": "Point", "coordinates": [t["lon"], t["lat"]]}
        }
        for t in targets
    ]
}

out = DATA_DIR / "targets.geojson"
with open(out, "w") as f:
    json.dump(targets_fc, f, indent=2)

print(f"Saved {len(targets_fc['features'])} targets → {out}")
for t in targets:
    print(f"  {t['name']:10}  ({t['lon']:>8.3f}, {t['lat']:>6.3f})  [path {t['path']}]")

Saved 4 targets → data\targets.geojson
  Tehran      (  51.388, 35.695)  [path Alpha]
  Honolulu    (-157.855, 21.305)  [path Bravo]
  Madrid      (  -3.703, 40.417)  [path Charlie]
  Riyadh      (  46.675, 24.688)  [path Delta]


## Exercise A

Create a 200 km buffer around **Honolulu** (`[-157.855, 21.305]`) and display it on a world map.

1. Call `buffer_feature` with `radius_km=200` and `name="Honolulu 200 km"`.
2. Add the point itself as a separate Feature.
3. Display both on a map centered on Honolulu at zoom 5.

In [7]:
from ipyleaflet import Map, GeoJSON, basemaps

# Honolulu coordinates
honolulu = [-157.855, 21.305]

# 1. Create 200 km buffer
buffer_feat = buffer_feature(
    *honolulu,
    radius_km=200,
    name="Honolulu 200 km"
)

# 2. Create point feature for Honolulu
point_feat = {
    "type": "Feature",
    "properties": {"name": "Honolulu"},
    "geometry": {
        "type": "Point",
        "coordinates": honolulu
    }
}

# FeatureCollection containing both
fc = {
    "type": "FeatureCollection",
    "features": [
        buffer_feat,
        point_feat
    ]
}

# 3. Display on map
m = Map(
    center=(honolulu[1], honolulu[0]),
    zoom=5,
    basemap=basemaps.CartoDB.Positron
)

m.add(GeoJSON(
    data=fc,
    style={
        "color": "#2980b9",
        "fillColor": "#2980b9",
        "fillOpacity": 0.25,
        "weight": 2
    }
))

m

Map(center=[21.305, -157.855], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zo…

## Exercise B

Create **three concentric buffers** (100 km, 250 km, 500 km) around **Madrid** (`[-3.703, 40.417]`) and display them in three different colors on a single map.

Print the number of vertices in each buffer's ring to confirm they are all using the default `n_points=64`.

In [8]:
from ipyleaflet import Map, GeoJSON, basemaps

madrid = [-3.703, 40.417]

radii = [100, 250, 500]
colors = {
    100: "#e74c3c",
    250: "#e67e22",
    500: "#f1c40f"
}

# Create buffers
buffers = {}

for radius in radii:
    feat = buffer_feature(
        *madrid,
        radius_km=radius,
        name=f"Madrid {radius} km"
    )
    buffers[radius] = feat

    # Print vertex count
    ring_count = len(feat["geometry"]["coordinates"][0])
    print(f"{radius} km buffer ring vertices: {ring_count}")


# Create map
m = Map(
    center=(madrid[1], madrid[0]),
    zoom=5,
    basemap=basemaps.CartoDB.Positron
)

# Display large → small so inner rings stay visible
for radius in [500, 250, 100]:
    m.add(GeoJSON(
        data={
            "type": "FeatureCollection",
            "features": [buffers[radius]]
        },
        name=f"Madrid {radius} km",
        style={
            "color": colors[radius],
            "fillColor": colors[radius],
            "fillOpacity": 0.25,
            "weight": 2
        }
    ))

# Add Madrid point
m.add(GeoJSON(
    data={
        "type": "FeatureCollection",
        "features": [
            {
                "type": "Feature",
                "properties": {"name": "Madrid"},
                "geometry": {
                    "type": "Point",
                    "coordinates": madrid
                }
            }
        ]
    },
    name="Madrid point"
))

m

100 km buffer ring vertices: 65
250 km buffer ring vertices: 65
500 km buffer ring vertices: 65


Map(center=[40.417, -3.703], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom…

---

## Check Your Understanding

**1.** What happens geometrically when you increase the buffer radius? What happens when you increase `n_points`?

**2.** Why are buffers stored as polygons in GeoJSON rather than as a center + radius pair?

```python
# No code needed — answer in your own words
```

## Check Your Understanding

### 1.
What happens geometrically when you increase the buffer radius? What happens when you increase `n_points`?

Increasing the buffer radius makes the buffer physically larger because all boundary points are placed farther from the center.

Examples:
- a 100 km buffer covers a smaller area
- a 500 km buffer covers a much larger area

Increasing `n_points` does not change the size of the buffer. Instead, it changes how smooth the polygon appears.

- Low `n_points` → more angular, rough shape
- High `n_points` → smoother circle approximation

For example:
- 8 points looks like an octagon
- 64 points looks much closer to a real circle

---

### 2.
Why are buffers stored as polygons in GeoJSON rather than as a center + radius pair?

GeoJSON does not have a native geometry type for circles.

Instead, GeoJSON stores shapes using explicit coordinates such as:
- Points
- LineStrings
- Polygons

Because of this, a buffer must be represented as a polygon made from many connected vertices.

Storing buffers as polygons also allows GIS software to:
- render them directly
- test intersections
- calculate overlap and area
- perform spatial operations consistently

A center + radius pair would describe the idea of a circle mathematically, but most GIS systems and GeoJSON tools expect actual polygon geometry for analysis.

## Next

In [02 — Buffering Lines](./02-Buffering_Lines.ipynb), we extend the point buffer to full line segments — creating a corridor zone around any path.